# Wpływ newsów na prognozę kierunku cen

Porównujemy ceny z cenami i sentymentem VADER na tym samym kalendarzu walk-forward. Najpierw zainstaluj `pip install -e ".[news,news-download,notebooks]"`. Źródła i ograniczenia są w `docs/NEWS.md`. Newsy nie stanowią gwarantowanego feedu point-in-time.

Audyt zachował tylko nagłówki Alphabet z datą w URL Nasdaq. Pilot: trening do 2019, test 2020. Dostępność tekstu jest przybliżeniem, a pełna historia AAPL/MSFT wymaga innego źródła.


In [ ]:
from pathlib import Path

import pandas as pd

from stock_direction.cli import main as run_experiment
from stock_direction.news import load_scored_news
from stock_direction.news_cli import main as prepare_news

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data/raw/news/fnspid_nasdaq100.csv"
SCORED = ROOT / "data/processed/news/vader_news.csv"

Pobieranie wymaga internetu. Zapisujemy pochodzenie i hash źródła; wybieramy wszystkie pasujące nagłówki, a nie pierwsze wiersze podglądu. Skrypty nie nadpisują istniejących plików.


In [ ]:
if not RAW.exists():
    prepare_news(["fetch-fnspid", "--output", str(RAW)])
if not SCORED.exists():
    prepare_news(["prepare", str(RAW), "--output", str(SCORED), "--require-url-date"])
news = load_scored_news(SCORED)
news.groupby("Ticker")["Available_At"].agg(["size", "min", "max"])

Domyślnie data newsa oznacza koniec jego dnia w Nowym Jorku, z dodatkowym opóźnieniem 24 godzin. Następnie dodajemy 5 minut i przypisujemy do pierwszego późniejszego zamknięcia giełdy. To zachowawcza reguła dla niezweryfikowanych czasów publikacji. Osobno sprawdź pokrycie każdego roku; brak feedu nie oznacza braku wydarzeń.


In [ ]:
coverage = (
    news.assign(Year=news["Available_At"].dt.year)
    .groupby(["Ticker", "Year"])
    .size()
    .unstack(0, fill_value=0)
)
coverage

Wybierz zakres testu po audycie pokrycia, przed oceną metryk. CLI zgłosi błąd, jeśli rok testowy nie ma newsów. Porównanie zapisuje prognozy, foldy, pokrycie i sparowany przedział zmiany accuracy.


In [ ]:
TICKER = "GOOGL"
FIRST_TRAIN_END_YEAR = 2019
LAST_TEST_YEAR = 2020
OUTPUT = ROOT / "reports/runs/news_notebook_googl"
run_experiment(
    [
        "--ticker",
        TICKER,
        "--models",
        "lr",
        "rf",
        "majority",
        "always_up",
        "--news-csv",
        str(SCORED),
        "--data-dir",
        str(ROOT / "data/raw/prices"),
        "--first-train-end-year",
        str(FIRST_TRAIN_END_YEAR),
        "--last-test-year",
        str(LAST_TEST_YEAR),
        "--output-dir",
        str(OUTPUT),
    ]
)
pd.read_csv(OUTPUT / "comparison.csv")

Dodatnia zmiana accuracy w pojedynczym eksperymencie wymaga osobnej weryfikacji i nie stanowi wyniku strategii. VADER jest ogólnym narzędziem słownikowym, nie modelem finansowym. Archiwum tekstów może zawierać późniejsze aktualizacje; reguła opóźnienia nie odtwarza historycznych rewizji. Stare notebooki `legacy` pozostają archiwum.
